%md
# A/B Test — Email Coupon Campaign

**Hypothesis:** Sending a 10%-off coupon email to existing one-time customers will increase the 30-day repurchase rate.

**Population:** Olist customers with exactly one prior order, last order ≥ 60 days ago (eligible for re-engagement).

**Design:** Two-group randomized test. 50/50 split between Treatment (gets email) and Control (no email). Fixed-duration test. No peeking, no early stopping.

**Primary metric:** 30-day repurchase rate (binary: did the customer place another order within 30 days of the assignment date?).

**Guardrails:** Average order value (AOV) of new orders — must not drop materially. Customer-complaint proxy (low review scores on repurchase) — must not spike.

**Note:** The campaign itself is simulated (we don't have a live email system). The methodology — power analysis, assignment, SRM check, statistical test, CI, guardrail evaluation — is identical to a production A/B test.


In [0]:
%pip install statsmodels

In [0]:
dbutils.library.restartPython()

In [0]:

# Imports — A/B Testing toolkit


# --- Data wrangling ---
import pandas as pd
import numpy as np

# --- Statistical analysis (the heart of A/B testing) ---
from scipy import stats
from statsmodels.stats.proportion import (
    proportions_ztest,              # z-test for two proportions (our primary metric is binary)
    proportion_confint,             # 95% CI for a single proportion
    confint_proportions_2indep,     # 95% CI for the DIFFERENCE between two proportions (the lift)
    proportion_effectsize,          # Cohen's h — effect size for proportions, used in power analysis
)
from statsmodels.stats.power import NormalIndPower  # power analysis for proportions

# --- Spark (read-only access to your existing gold tables) ---
from pyspark.sql import functions as F

# --- Visualization ---
import matplotlib.pyplot as plt

# --- Reproducibility ---
import random
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# --- Sanity check ---
print("✓ All imports loaded.")
print(f"✓ pandas {pd.__version__}, numpy {np.__version__}")
print(f"✓ Random seed fixed at {RANDOM_SEED} (results will be reproducible).")

%md
## 1. Hypothesis & Metric Design

### Business hypothesis
Sending a 10%-off coupon email to existing one-time customers will increase
the 30-day repurchase rate by at least **1 percentage point**.

### Statistical hypothesis
- **Null (H₀):** p_treatment = p_control — the email has no effect.
- **Alternative (H₁):** p_treatment ≠ p_control — the email changes the rate (either direction).

We use a **two-sided** test because we want to detect harm as well as benefit.

### Primary metric
**30-day repurchase rate** = (customers who placed another order within 30 days of assignment) / (customers in the group)

- Binary per customer: 1 = repurchased, 0 = did not.
- Aggregated as a proportion per group.
- Why this metric: it directly captures the campaign's stated goal — re-engaging one-time buyers.

### Guardrail metrics
1. **Average Order Value (AOV)** on new orders — must not drop more than 5%. Protects margin: a 10% coupon that drags basket size down 20% is a net loss.
2. **Review score** on repurchase orders — must stay ≥ 4.0. Protects customer satisfaction.

### Decision rule
| Outcome | Decision |
|---|---|
| Primary lift significant (p<0.05) AND ≥0.5pp AND no guardrail breach | **SHIP** to full base |
| Primary lift not significant OR negative | **KILL** |
| Primary lift significant but <0.5pp | **ITERATE** — test stronger offer |
| Any guardrail breach | **KILL**, regardless of primary |

In [0]:

# Test configuration — every parameter in one place
# This is the "intake template" pattern: a real experimentation
# framework would generate this from a form the marketer fills in.


TEST_CONFIG = {
    # Identification
    "test_name":        "email_coupon_v1",
    "hypothesis":       "10% off coupon email lifts 30-day repurchase rate by >=1pp",

    # Statistical parameters
    "alpha":            0.05,    # significance threshold (false-positive rate)
    "power":            0.80,    # statistical power (1 - false-negative rate)
    "baseline_rate":    0.035,   # assumed control repurchase rate (3.5%)
    "mde_pp":           0.01,    # minimum detectable effect = 1 percentage point
    "test_type":        "two-sided",

    # Test design
    "split":            0.50,    # 50/50 treatment vs control
    "duration_days":    30,      # measurement window after assignment

    # Decision rule
    "min_practical_lift_pp": 0.005,  # 0.5 pp minimum lift to ship
    "aov_guardrail_pct":     -0.05,  # AOV can drop max 5%
    "review_guardrail":      4.0,    # review score must stay >= 4.0
}

# Pretty-print
print("┌─ TEST CONFIGURATION ──────────────────────────────────")
for k, v in TEST_CONFIG.items():
    print(f"│  {k:.<30} {v}")
print("└───────────────────────────────────────────────────────")

%md
## 2. Power Analysis
Calculate the required sample size for a two-sided test of two proportions,
using Cohen's h as the effect size measure.

In [0]:

# Power Analysis — how big does the sample need to be?


p0 = TEST_CONFIG["baseline_rate"]            # control rate: 3.5%
p1 = p0 + TEST_CONFIG["mde_pp"]              # treatment rate if MDE is real: 4.5%

# Cohen's h — computed manually with numpy. The formula is:
#   h = 2 * (arcsin(sqrt(p1)) - arcsin(sqrt(p0)))
# This avoids any library function-naming issues entirely.
effect_h = 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p0)))

# Solve for required sample size per group
analysis = NormalIndPower()
n_per_group = analysis.solve_power(
    effect_size = effect_h,
    alpha       = TEST_CONFIG["alpha"],
    power       = TEST_CONFIG["power"],
    alternative = "two-sided",
)
n_per_group = int(np.ceil(n_per_group))
n_total = n_per_group * 2

# Save back to config so later cells can use it
TEST_CONFIG["n_per_group"] = n_per_group
TEST_CONFIG["n_total"]     = n_total

# Pretty-print
mag = "small" if abs(effect_h) < 0.2 else "medium" if abs(effect_h) < 0.5 else "large"
print("┌─ POWER ANALYSIS ──────────────────────────────────────")
print(f"│  Baseline rate (p0):       {p0:.1%}")
print(f"│  Target rate  (p1):        {p1:.1%}   (= p0 + MDE)")
print(f"│  Cohen's h (effect size):  {effect_h:.4f}   ({mag} effect)")
print(f"│  Alpha:                    {TEST_CONFIG['alpha']}")
print(f"│  Power:                    {TEST_CONFIG['power']:.0%}")
print(f"│  Test type:                {TEST_CONFIG['test_type']}")
print("├───────────────────────────────────────────────────────")
print(f"│  REQUIRED per group:       {n_per_group:>8,}")
print(f"│  REQUIRED total:           {n_total:>8,}")
print("└───────────────────────────────────────────────────────")

In [0]:

# Sensitivity: how sample size scales with the MDE we care about


print("┌─ SAMPLE-SIZE SENSITIVITY TO MDE ──────────────")
print(f"│  Baseline {p0:.1%} · alpha={TEST_CONFIG['alpha']} · power={TEST_CONFIG['power']:.0%}")
print("├───────────────────────────────────────────────")
print("│   MDE (pp)   Relative lift   n per group")
print("├───────────────────────────────────────────────")
for mde in [0.005, 0.0075, 0.01, 0.015, 0.02, 0.03]:
    p_alt = p0 + mde
    h = 2 * (np.arcsin(np.sqrt(p_alt)) - np.arcsin(np.sqrt(p0)))
    n = int(np.ceil(NormalIndPower().solve_power(
        effect_size=h, alpha=TEST_CONFIG['alpha'],
        power=TEST_CONFIG['power'], alternative='two-sided')))
    print(f"│    {mde*100:>5.2f}     {(mde/p0)*100:>6.1f}%      {n:>10,}")
print("└───────────────────────────────────────────────")

%md
## 3. Customer Eligibility & Random Assignment

### Eligibility criteria (a re-engagement test)
| Criterion | Value | Why |
|---|---|---|
| Order count so far | exactly 1 | One-time buyers — the 96% segment we want to re-engage |
| Days since last order | 60 to 180 | Quiet but not gone — a natural re-engagement window |
| Reference "today" | 2018-09-01 | Cutoff date used as our test-design "today" |

### Assignment
- **Sample size:** 12,010 (from power analysis)
- **Split:** 50/50 random, seed-controlled
- **Mode:** in-memory pandas — no new tables written to the catalog (safe)

In [0]:

# Load eligible customers from silver enriched layer
# READ-ONLY — does NOT write anything back to the catalog.


CUTOFF_DATE = "2018-09-01"

eligibility_query = f"""
WITH customer_history AS (
    SELECT
        customer_unique_id,
        COUNT(DISTINCT order_id) AS order_count,
        MAX(DATE(order_purchase_timestamp)) AS last_order_date
    FROM olist.silver.orders_enriched
    WHERE DATE(order_purchase_timestamp) <= DATE('{CUTOFF_DATE}')
    GROUP BY customer_unique_id
)
SELECT
    customer_unique_id,
    last_order_date,
    DATEDIFF(DATE('{CUTOFF_DATE}'), last_order_date) AS days_since_last_order
FROM customer_history
WHERE order_count = 1
  AND DATEDIFF(DATE('{CUTOFF_DATE}'), last_order_date) BETWEEN 60 AND 180
"""

# Spark runs the filter; only the (small) eligible result lands in pandas.
eligible_sdf = spark.sql(eligibility_query)
eligible_df  = eligible_sdf.toPandas()

print(f"Eligible customers (one-time, 60-180d quiet):  {len(eligible_df):,}")
print(f"Required for test:                             {TEST_CONFIG['n_total']:,}")
print()
if len(eligible_df) >= TEST_CONFIG['n_total']:
    print(f"✓ Pool is large enough — we'll sample {TEST_CONFIG['n_total']:,} from {len(eligible_df):,}.")
else:
    print(f"⚠ Pool too small ({len(eligible_df):,} < {TEST_CONFIG['n_total']:,}).")
    print(f"  Options: widen the 60-180d window, or accept lower test power.")

print("\nDays-since-last-order distribution in eligible pool:")
print(eligible_df['days_since_last_order'].describe().round(1))

In [0]:

# Sample n_total customers + assign 50/50 to Treatment/Control
# Random seed makes this fully reproducible.


# 1. Sample n_total from the eligible pool
sampled_df = eligible_df.sample(
    n=TEST_CONFIG["n_total"],
    random_state=RANDOM_SEED
).reset_index(drop=True)

# 2. Random assignment: half Treatment, half Control, then shuffle
half = TEST_CONFIG["n_per_group"]
sampled_df["group"] = ["treatment"] * half + ["control"] * half

# 3. Shuffle so DataFrame position doesn't correlate with group
assignment_df = sampled_df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

# 4. Assignment summary
n_t = (assignment_df["group"] == "treatment").sum() 
n_c = (assignment_df["group"] == "control").sum()

print("┌─ ASSIGNMENT SUMMARY ──────────────────────────────")
print(f"│  Total assigned:    {len(assignment_df):>8,}")
print(f"│  Treatment:         {n_t:>8,}   ({n_t/len(assignment_df):.2%})")
print(f"│  Control:           {n_c:>8,}   ({n_c/len(assignment_df):.2%})")
print(f"│  Target split:      50.00% / 50.00%")
print("└───────────────────────────────────────────────────")

print("\nFirst 5 assignments:")
display(assignment_df.head())

%md
## 4. SRM Check (Sample Ratio Mismatch)

Chi-square test: does the observed split match the intended 50/50 split?

**Interpretation rule:**
- p ≥ 0.05  →  ✓ assignment looks fine
- 0.01 ≤ p < 0.05  →  ⚠ borderline, investigate
- p < 0.01  →  ✗ STOP the test, randomization is broken

In [0]:

# SRM Check — does the actual split match the intended 50/50?


from scipy.stats import chisquare

n_t = int((assignment_df["group"] == "treatment").sum())
n_c = int((assignment_df["group"] == "control").sum())
total = n_t + n_c

expected_t = total * TEST_CONFIG["split"]
expected_c = total * (1 - TEST_CONFIG["split"])

chi2, p_value = chisquare(f_obs=[n_t, n_c], f_exp=[expected_t, expected_c])

print("┌─ SRM CHECK (Sample Ratio Mismatch) ───────────────")
print(f"│  Observed:    treatment={n_t:>6,}    control={n_c:>6,}")
print(f"│  Expected:    treatment={expected_t:>6,.0f}    control={expected_c:>6,.0f}")
print(f"│  Chi-square:  {chi2:.4f}")
print(f"│  p-value:     {p_value:.4f}")
print("├───────────────────────────────────────────────────")
if p_value < 0.01:
    verdict = "✗ FAIL — Severe SRM. STOP the test, fix randomization."
elif p_value < 0.05:
    verdict = "⚠ WARN — Borderline SRM. Investigate before trusting results."
else:
    verdict = "✓ PASS — Assignment ratio matches the 50/50 design."
print(f"│  {verdict}")
print("└───────────────────────────────────────────────────")

In [0]:

# Covariate Balance Check — were the two groups similar BEFORE the test?
# If randomization worked, days_since_last_order should look identical
# in treatment and control. If not, randomization had a leak.


from scipy.stats import ttest_ind

t_days = assignment_df.loc[assignment_df["group"] == "treatment", "days_since_last_order"]
c_days = assignment_df.loc[assignment_df["group"] == "control",   "days_since_last_order"]

# Two-sample t-test on the covariate
t_stat, p_balance = ttest_ind(t_days, c_days, equal_var=False)

print("┌─ COVARIATE BALANCE — days_since_last_order ───────")
print(f"│  Treatment   mean={t_days.mean():>6.1f}   std={t_days.std():>5.1f}   n={len(t_days):,}")
print(f"│  Control     mean={c_days.mean():>6.1f}   std={c_days.std():>5.1f}   n={len(c_days):,}")
print(f"│  Diff in means:  {t_days.mean() - c_days.mean():+.2f} days")
print(f"│  t-test p-value: {p_balance:.4f}")
print("├───────────────────────────────────────────────────")
if p_balance < 0.05:
    print("│  ⚠ WARN — Groups differ on this covariate.")
    print("│        Random chance? Re-run with a different seed and re-check.")
else:
    print("│  ✓ PASS — Groups are balanced on days_since_last_order.")
print("└───────────────────────────────────────────────────")

%md
## 5. Simulate Campaign Outcomes

**True parameters (we'd never know these in a real test):**

| Metric | Control | Treatment | True lift |
|---|---|---|---|
| Repurchase rate | 3.5% | 4.5% | +1pp |
| AOV | R$ 150 | R$ 145 | −R$ 5 |
| Review score | 4.2 | 4.2 | 0 |

We simulate outcomes from these true parameters, then in the next step
we'll analyze the data *as if we didn't know them* — to validate that
our analysis correctly recovers the effect.

In [0]:

# Simulate Campaign Outcomes
# (In a real test, these would come from your email + commerce logs.
#  Here we generate them from known TRUE parameters so we can validate
#  that our analysis correctly recovers the effect.)


# TRUE parameters — these are the "hidden" answers we want our test to recover
TRUE_CONTROL_RATE   = 0.035    # baseline 3.5%
TRUE_TREATMENT_RATE = 0.045    # +1pp lift (= our MDE)
TRUE_CONTROL_AOV    = 150.0
TRUE_TREATMENT_AOV  = 145.0    # small drop from the coupon discount
TRUE_REVIEW_MEAN    = 4.2

# Use a different seed so the outcomes don't accidentally correlate with assignment
np.random.seed(RANDOM_SEED + 1)

# Initialize outcome columns
assignment_df["repurchased"]  = 0
assignment_df["aov"]          = np.nan
assignment_df["review_score"] = np.nan

# --- Repurchase outcomes (Bernoulli per group) ---
t_mask = assignment_df["group"] == "treatment"
c_mask = assignment_df["group"] == "control"

assignment_df.loc[t_mask, "repurchased"] = np.random.binomial(1, TRUE_TREATMENT_RATE, size=t_mask.sum())
assignment_df.loc[c_mask, "repurchased"] = np.random.binomial(1, TRUE_CONTROL_RATE,   size=c_mask.sum())

# --- For repurchasers only: simulate AOV and review score ---
rep_t = t_mask & (assignment_df["repurchased"] == 1)
rep_c = c_mask & (assignment_df["repurchased"] == 1)

# AOV ~ Normal, clipped at R$ 10 minimum
assignment_df.loc[rep_t, "aov"] = np.clip(
    np.random.normal(TRUE_TREATMENT_AOV, 50, size=rep_t.sum()), 10, None
)
assignment_df.loc[rep_c, "aov"] = np.clip(
    np.random.normal(TRUE_CONTROL_AOV, 50, size=rep_c.sum()), 10, None
)

# Review score ~ Normal around 4.2, clipped to 1..5, rounded to integer
rep_all = assignment_df["repurchased"] == 1
assignment_df.loc[rep_all, "review_score"] = np.round(
    np.clip(np.random.normal(TRUE_REVIEW_MEAN, 0.7, size=rep_all.sum()), 1, 5)
)

# --- Summary ---
print("┌─ SIMULATION COMPLETE ─────────────────────────────")
print(f"│  Repurchasers in treatment:  {rep_t.sum():>5,}   (raw rate {rep_t.sum()/t_mask.sum():.2%})")
print(f"│  Repurchasers in control:    {rep_c.sum():>5,}   (raw rate {rep_c.sum()/c_mask.sum():.2%})")
print(f"│  Observed difference:        {rep_t.sum()/t_mask.sum() - rep_c.sum()/c_mask.sum():+.4f} pp")
print(f"│                              (true diff was +0.0100 pp)")
print("└───────────────────────────────────────────────────")

print("\nSample of simulated outcomes:")
display(assignment_df.head(10))

## 6. Primary Metric Analysis

Two-proportions z-test on 30-day repurchase rate.
Decision rule (pre-registered): SHIP if statistically significant (p<0.05) AND practically significant (lift ≥ 0.5 pp) AND no guardrail breach.

In [0]:

# Primary Metric Analysis — z-test for two proportions + CI on the lift

# Per-group counts (treat the simulated outcomes as if they were real)
n_t = int(t_mask.sum())           # total treatment customers
n_c = int(c_mask.sum())           # total control customers
x_t = int(assignment_df.loc[t_mask, "repurchased"].sum())        # total repurchases in treatment
x_c = int(assignment_df.loc[c_mask, "repurchased"].sum())        # total repurchases in control

p_t = x_t / n_t           # treatment repurchase rate
p_c = x_c / n_c           # control repurchase rate
abs_lift = p_t - p_c       # absolute lift in percentage points
rel_lift = abs_lift / p_c     # relative lift (percentage of baseline)

# Two-proportions z-test (two-sided)
counts = np.array([x_t, x_c])
nobs   = np.array([n_t, n_c])   # number of observations (for each group)  
z_stat, p_value = proportions_ztest(counts, nobs, alternative="two-sided")

# 95% CI for the DIFFERENCE in proportions
# (using Wald method — fine for our sample size; Newcomb is more robust for small samples)
ci_low, ci_high = confint_proportions_2indep(
    count1=x_t, nobs1=n_t,
    count2=x_c, nobs2=n_c,
    method="wald",
    compare="diff"
)

# Decisions
stat_sig      = p_value < TEST_CONFIG["alpha"]
practical_sig = abs_lift >= TEST_CONFIG["min_practical_lift_pp"]
ci_excludes_0 = (ci_low > 0) or (ci_high < 0)

print("┌─ PRIMARY METRIC ANALYSIS ─────────────────────────────")
print(f"│  Control rate:           {p_c:.4f}    ({p_c:.2%})")
print(f"│  Treatment rate:         {p_t:.4f}    ({p_t:.2%})")
print("├────────────────────────────────────────────────────────")
print(f"│  Absolute lift:          {abs_lift:+.4f}    ({abs_lift*100:+.2f} pp)")
print(f"│  Relative lift:          {rel_lift:+.2%}")
print(f"│  95% CI on absolute lift: [{ci_low:+.4f}, {ci_high:+.4f}]")
print(f"│                           [{ci_low*100:+.2f} pp, {ci_high*100:+.2f} pp]")
print("├────────────────────────────────────────────────────────")
print(f"│  z-statistic:            {z_stat:.4f}")
print(f"│  p-value:                {p_value:.4f}")
print("├────────────────────────────────────────────────────────")
print(f"│  Statistical significance: {'✓ YES' if stat_sig else '✗ NO'}  (p < {TEST_CONFIG['alpha']})")
print(f"│  Practical significance:   {'✓ YES' if practical_sig else '✗ NO'}  (lift ≥ {TEST_CONFIG['min_practical_lift_pp']*100:.1f} pp)")
print(f"│  CI excludes zero:         {'✓ YES' if ci_excludes_0 else '✗ NO'}")
print("└────────────────────────────────────────────────────────")

## 7. Guardrail Analysis & Final Scorecard

Two guardrails enforced:
1. **AOV** — average order value of repurchase orders. Must not drop more than 5%.
2. **Review score** — average review of repurchase orders. Must stay ≥ 4.0.

A breach on either guardrail = KILL the campaign, regardless of primary lift.

In [0]:

# Guardrail Analysis — AOV and Review Score


# --- Guardrail 1: Average Order Value (AOV) ---
aov_t = assignment_df.loc[rep_t, "aov"].dropna()
aov_c = assignment_df.loc[rep_c, "aov"].dropna()

aov_t_mean   = aov_t.mean()
aov_c_mean   = aov_c.mean()
aov_diff_abs = aov_t_mean - aov_c_mean
aov_diff_pct = aov_diff_abs / aov_c_mean

t_stat_aov, p_aov = stats.ttest_ind(aov_t, aov_c, equal_var=False)
aov_breach = aov_diff_pct < TEST_CONFIG["aov_guardrail_pct"]

# --- Guardrail 2: Review Score ---
rev_t = assignment_df.loc[rep_t, "review_score"].dropna()
rev_c = assignment_df.loc[rep_c, "review_score"].dropna()
rev_t_mean = rev_t.mean()
rev_c_mean = rev_c.mean()
review_breach = rev_t_mean < TEST_CONFIG["review_guardrail"]

print("┌─ GUARDRAIL ANALYSIS ──────────────────────────────────")
print(f"│  AOV — Treatment:        R$ {aov_t_mean:>7.2f}   (n={len(aov_t)})")
print(f"│  AOV — Control:          R$ {aov_c_mean:>7.2f}   (n={len(aov_c)})")
print(f"│  AOV difference:         R$ {aov_diff_abs:>+7.2f}   ({aov_diff_pct:+.2%})")
print(f"│  AOV t-test p-value:     {p_aov:.4f}")
print(f"│  AOV threshold:          drop must be ≤ {abs(TEST_CONFIG['aov_guardrail_pct']):.0%}")
print(f"│  AOV verdict:            {'✗ BREACH' if aov_breach else '✓ OK'}")
print("├────────────────────────────────────────────────────────")
print(f"│  Review — Treatment:     {rev_t_mean:>5.2f}   (n={len(rev_t)})")
print(f"│  Review — Control:       {rev_c_mean:>5.2f}   (n={len(rev_c)})")
print(f"│  Review threshold:       Treatment ≥ {TEST_CONFIG['review_guardrail']}")
print(f"│  Review verdict:         {'✗ BREACH' if review_breach else '✓ OK'}")
print("└────────────────────────────────────────────────────────")

# Save for scorecard
GUARDRAIL_RESULT = {
    "aov_t": aov_t_mean,
    "aov_c": aov_c_mean,
    "aov_pct_change": aov_diff_pct,
    "aov_breach": aov_breach,
    "review_t": rev_t_mean,
    "review_breach": review_breach,
    "any_breach": bool(aov_breach or review_breach),
}

In [0]:

# Final Executive Scorecard — one-page ship/kill decision


# Decide the verdict based on the pre-registered rule
if GUARDRAIL_RESULT["any_breach"]:
    DECISION  = "KILL"
    RATIONALE = "Guardrail breach — never ship regardless of primary metric."
elif not stat_sig:
    DECISION  = "KILL or RE-TEST"
    RATIONALE = "Primary metric not statistically significant — no evidence the email works."
elif not practical_sig:
    DECISION  = "ITERATE"
    RATIONALE = "Lift is significant but below practical threshold — test a stronger offer."
else:
    DECISION  = "SHIP"
    RATIONALE = "Significant lift, above practical threshold, no guardrail breach."

print("=" * 67)
print("  A/B TEST SCORECARD")
print(f"  Test: {TEST_CONFIG['test_name']}")
print("=" * 67)
print()
print(f"  Hypothesis:   {TEST_CONFIG['hypothesis']}")
print(f"  Sample:       {n_t + n_c:,} customers, 50/50 split")
print(f"  Duration:     {TEST_CONFIG['duration_days']} days")
print()
print("-" * 67)
print("  PRIMARY METRIC — 30-day repurchase rate")
print("-" * 67)
print(f"    Control rate:        {p_c:>7.4f}   ({p_c:>6.2%})")
print(f"    Treatment rate:      {p_t:>7.4f}   ({p_t:>6.2%})")
print(f"    Absolute lift:       {abs_lift*100:>+7.2f} pp")
print(f"    Relative lift:       {rel_lift:>+7.2%}")
print(f"    95% CI on lift:      [{ci_low*100:+.2f} pp, {ci_high*100:+.2f} pp]")
print(f"    p-value:             {p_value:.4f}")
print(f"    Statistical sig:     {'✓ YES' if stat_sig else '✗ NO'}")
print(f"    Practical sig:       {'✓ YES' if practical_sig else '✗ NO'}")
print()
print("-" * 67)
print("  GUARDRAILS")
print("-" * 67)
print(f"    AOV % change:        {GUARDRAIL_RESULT['aov_pct_change']:>+.2%}     {'✓ OK' if not GUARDRAIL_RESULT['aov_breach'] else '✗ BREACH'}")
print(f"    Review score (T):    {GUARDRAIL_RESULT['review_t']:>5.2f}       {'✓ OK' if not GUARDRAIL_RESULT['review_breach'] else '✗ BREACH'}")
print()
print("=" * 67)
print(f"  DECISION:    {DECISION}")
print(f"  Rationale:   {RATIONALE}")
print("=" * 67)

## 8. Visualization & Recommendation

In [0]:

# Repurchase rate by group, with 95% Wilson CI error bars
# (Slide-ready: clean, single message, color-coded by treatment.)


import matplotlib.pyplot as plt

# Per-group CIs for individual proportions (Wilson method = robust for proportions)
ci_c_low, ci_c_high = proportion_confint(x_c, n_c, alpha=0.05, method="wilson")
ci_t_low, ci_t_high = proportion_confint(x_t, n_t, alpha=0.05, method="wilson")

fig, ax = plt.subplots(figsize=(8, 5.5))

groups = ["Control", "Treatment"]
rates  = [p_c, p_t]
err_lower = [p_c - ci_c_low, p_t - ci_t_low]
err_upper = [ci_c_high - p_c, ci_t_high - p_t]
colors = ["#9CA3AF", "#1F4E79"]

bars = ax.bar(
    groups, rates,
    yerr=[err_lower, err_upper],
    capsize=10, color=colors, edgecolor='black', linewidth=0.8, width=0.55
)

# Rate labels on top of bars
for bar, rate in zip(bars, rates):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.002,
        f"{rate:.2%}", ha='center', va='bottom',
        fontsize=12, fontweight='bold'
    )

# Title with the headline finding
sig_marker = "✓ significant" if stat_sig else "✗ not significant"
ax.set_title(
    f"Email Coupon A/B Test:  +{abs_lift*100:.2f} pp lift  ({rel_lift:+.0%}),  p = {p_value:.4f}  {sig_marker}",
    fontsize=12, fontweight='bold', pad=15, color="#1F4E79"
)
ax.set_ylabel("30-day repurchase rate", fontsize=11)

# Subtitle with sample info
ax.text(
    0.5, -0.16,
    f"n = {n_t + n_c:,} customers (50/50 split)  ·  Error bars = 95% Wilson CI",
    transform=ax.transAxes, ha='center', fontsize=9, color="#6B7280", style='italic'
)

# Styling
ax.set_ylim(0, max(rates) * 1.5)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.25, linestyle='--')

plt.tight_layout()
plt.show()


## Recommendation

**Decision:** SHIP the email coupon campaign to the full eligible base of one-time customers who are 60–180 days quiet.

**Why:**
- Statistically significant **+1.05 pp lift** in 30-day repurchase rate (29% relative), 95% CI of **+0.34 pp to +1.76 pp**. Even at the lower bound, the campaign adds ~10% relative repurchases on top of baseline.
- Both pre-registered guardrails — average order value and review score — remained within tolerance.
- The result is consistent with the test design: we powered to detect a 1 pp lift, and we detected one.

**What to watch post-launch:**
- **Cannibalization risk.** Some of the +1 pp may pull demand forward rather than create it. Recommend a 90-day post-launch cohort comparison vs a permanent holdout to estimate true incrementality.
- **AOV drift.** Our test showed AOV roughly stable but the sample size on AOV was small (n≈250 per group), so the CI on AOV is wide. Monitor weekly post-launch.
- **Audience fatigue.** Limit re-sends to one coupon per customer per quarter.

**Next experiments to run:**
1. **Coupon-depth test** — 10% vs 15% vs 20%. Find the discount level that maximizes incremental margin (not just incremental revenue).
2. **Send-time test** — same coupon, different time of day. Cheap to test, often yields meaningful lift.
3. **Holdout discipline** — keep a permanent ~5% holdout from all CRM sends to estimate long-term program ROI.